In [106]:
import google.generativeai as genai
import json
from typing import List, Dict, Optional
import time
import pandas as pd

In [107]:

# Configure Gemini API
def setup_gemini(api_key: str):
    genai.configure(api_key=api_key)
    return genai.GenerativeModel('gemini-2.0-flash')

In [108]:

test_df = pd.read_json('../Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl', lines=True).to_dict(orient='records')
ground_truth = []
for val in test_df:
    ground_truth.append(val.get('text_a_is_closer'))

In [109]:
SYSTEM_PROMPT = """
You are an expert in narrative understanding. 
Given an anchor story and two candidate stories (Text A and Text B), decide which candidate is more narratively similar to the anchor.

Assess similarity based on:
- **Theme** – underlying ideas, goals, or moral.
- **Event progression** – key actions and turning points.
- **Causality and timeline** – how and why events unfold.
- **Character roles** – who acts, what motivates them, and their outcomes.
- **Resolution** – how conflicts or story arcs conclude.

Focus on overall narrative structure and meaning, not surface wording or setting details.

Respond with only one word:
- true  → if Text A is more similar to the anchor
- false → if Text B is more similar
"""


In [110]:
def create_prompt(anchor_text,text_a, text_b, examples= None):
    """Create the prompt with optional few-shot examples"""
    prompt = SYSTEM_PROMPT + "\n\n"
    
    if examples:
        prompt += "Here are some examples:\n\n"
        for i, ex in enumerate(examples, 1):
            prompt += f"Example {i}:\n"
            prompt += f"Anchor: {ex['anchor_text']}\n"
            prompt += f"Text A: {ex['text_a']}\n"
            prompt += f"Text B: {ex['text_b']}\n"
            prompt += f"Answer: {str(ex['text_a_is_closer']).lower()}\n\n"
    
    prompt += "Now analyze this case:\n"
    prompt += f"Anchor: {anchor_text}\n"
    prompt += f"Text A: {text_a}\n"
    prompt += f"Text B: {text_b}\n"
    prompt += "Answer (true/false):"
    
    return prompt

In [111]:
def parse_response(response_text: str) -> bool:
    """Parse the model response to extract boolean answer"""
    response_text = response_text.strip().lower()
    
    # Handle various response formats
    if 'true' in response_text:
        return True
    elif 'false' in response_text:
        return False
    else:
        # Fallback: if unclear, default to True
        print(f"Warning: Unclear response: {response_text}")
        return True

In [112]:
def predict_similarity(model,anchor,text_a,text_b,examples = None,temperature= 0.0):
    """Make a single prediction"""
    prompt = create_prompt(anchor, text_a, text_b, examples)
    
    generation_config = genai.types.GenerationConfig(
        temperature=temperature,
        max_output_tokens=10,
    )
    
    try:
        response = model.generate_content(
            prompt,
            generation_config=generation_config
        )
        return parse_response(response.text)
    except Exception as e:
        print(f"Error during prediction: {e}")
        return True  # Default fallback

In [113]:
def batch_predict(model,test_data_path,examples = None,temperature= 0.0,sleep_time= 1.0):

    results = []
    test_data = pd.read_json(test_data_path, lines=True).to_dict(orient='records')
    for i, item in enumerate(test_data):
        if(i%20 == 0):
            print(f"Processed {i+1}/{len(test_data)}: {item.get('id', i)}")
        
        prediction = predict_similarity(
            model,
            anchor=item['anchor_text'],
            text_a=item['text_a'],
            text_b=item['text_b'],
            examples=examples,
            temperature=temperature
        )
        
        results.append({
            'id': item.get('id', i),
            'anchor': item['anchor_text'],
            'text_a': item['text_a'],
            'text_b': item['text_b'],
            'text_a_is_closer': prediction
        })
        
        # Rate limiting
        if i < len(test_data) - 1:
            time.sleep(sleep_time)
        
    
    return results

In [114]:


def save_results(results: List[Dict], output_path: str):
    """Save predictions to file"""
    with open(output_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to {output_path}")


In [115]:
import pandas as pd
few_shot_examples=[]
df = pd.read_json('../Data/SemEval2026-Task_4-sample-v1/sample_track_a.jsonl', lines=True)
few_shot_examples = []
few_shot_shortlisted_examples = df.sample(n=3)
for index, row in few_shot_shortlisted_examples.iterrows():
    current_example={}
    for column_name, value in row.items():
        current_example['anchor_text']=row['anchor_text']
        current_example['text_a']=row['text_a']
        current_example['text_b']=row['text_b']
        current_example['text_a_is_closer']=row['text_a_is_closer']
    few_shot_examples.append(current_example)
few_shot_examples

[{'anchor_text': 'In 1711 Usbek leaves his seraglio in Isfahan to make the long journey to France, accompanied by his young friend Rica. He leaves behind five wives (Zashi, Zéphis, Fatmé, Zélis, and Roxane) in the care of a number of black eunuchs, one of whom is the head or first eunuch. During the trip and their long stay in Paris (1712 to 1720), they comment, in letters exchanged with friends and mullahs, on numerous aspects of Western, Christian society, particularly French politics and manners, including a biting satire of the System of John Law. Over time, various disorders surface back in the seraglio, and, beginning in 1717 (Letter 139 [147]), that situation rapidly unravels. Usbek orders his head eunuch to crack down, but his message does not arrive in time, and the internal revolt brings about the death of his wives, including the vengeful suicide of his favorite, Roxane, and, it appears, most of the eunuchs.\nThe Chronology can be summarized as follows:',
  'text_a': "Three 

In [ ]:
API_KEY = ""

model = setup_gemini(API_KEY)


# 0-shot prediction
print("Running 0-shot predictions...")
results_0shot = batch_predict(
    model,
    test_data_path='../Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl',
    examples=None,
    temperature=0.0
)

accuracy=0
for pred,actual in zip(results_0shot, ground_truth):
    if(pred['text_a_is_closer'] == actual):
        accuracy += 1
print("0 shot prompting accuracy: ", accuracy/len(ground_truth))




Running 0-shot predictions...
Processed 1/200: 0
Processed 21/200: 20
Processed 41/200: 40
Processed 61/200: 60
Processed 81/200: 80
Processed 101/200: 100
Processed 121/200: 120
Processed 141/200: 140
Processed 161/200: 160
Processed 181/200: 180
0 shot prompting accuracy:  0.635


In [117]:
# Few-shot prediction
print("\nRunning few-shot predictions...")
results_fewshot = batch_predict(
    model,
    test_data_path='../Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl',
    examples=few_shot_examples,
    temperature=0.0
)

accuracy=0
for pred,actual in zip(results_fewshot, ground_truth):
    if(pred['text_a_is_closer'] == actual):
        accuracy += 1
print("few shot prompting accuracy: ", accuracy/len(ground_truth))




Running few-shot predictions...
Processed 1/200: 0
Processed 21/200: 20
Processed 41/200: 40
Processed 61/200: 60
Processed 81/200: 80
Processed 101/200: 100
Processed 121/200: 120
Processed 141/200: 140
Processed 161/200: 160
Processed 181/200: 180
few shot prompting accuracy:  0.615
